σ-timed OCO breakouts on XAUUSD

Q1 (precondition): Does a run of small candles on the 15- or 30-minute
chart predict a larger move over the next H minutes than the HAR
volatility model already forecasts? 'Small' is measured relative to σ,
so the pattern means the same in calm and volatile years.

Q2 (strategy): When the HAR model and/or the small-candle pattern
indicate a quiet market expected to expand, do symmetric stop-entry
(OCO) breakout orders give better net outcomes than the same orders at
untimed squeezes or at random times?

Motivation: the owner's discretionary observation that compressed
periods of tiny candles tend to precede strong moves. This motivates the
hypothesis; it is not evidence for it.

Status: EXPLORATORY, Stage 1. Not a PREREG. Nothing here is evidence.
Q2 is only pursued if Q1 shows the pattern adds information.

## How this notebook works

- **Register first.** Every number, threshold, window, cost, seed or definition comes from the register cell below.
  Code reads a value only through `need(id)`, which raises unless the entry is **APPROVED**. A cell whose inputs aren't
  all approved prints what blocks it and computes nothing.
- **Guards.** Every frame is checked against the dev period with the repo's seal guards (`research/sealing.py`,
  `research/vol_features_r3/run_r3.py` `assert_sealed`). Nothing outside 2007-01-01 .. 2020-12-31 is read.
- **The work is the repo's code.** `stage1.py` (next to this notebook) only wires the repo's loader, `vol_block`
  and `sigma_compare`'s post-hoc scoring together, with values passed in from the register. Its tests:
  `research/squeeze_oco/test_stage1.py` (synthetic bars).
- **Data licence.** `XAUUSD.txt` is IBKR data, licensed for the owner's own use only. Outputs show aggregates and
  charts only, never a price row.
- **Record keeping.** The decision log below is append-only. `ATTEMPTS.json` counts outcome analyses; Stage 1 looks
  at no outcome and doesn't count.

In [1]:
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
if not (HERE / "stage1.py").exists():                    # run from the repo root
    HERE = Path("research/squeeze_oco").resolve()
sys.path.insert(0, str(HERE))
import stage1  # noqa: E402

pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 80, "display.max_colwidth", 400)

## Register

`status`: **APPROVED** (usable), **PROPOSED** (a value with reasoning, waiting for the owner), **OPEN** (no value yet).
`source`: a repo file and line, or "owner, date". Changing an entry is a decision-log line.

In [2]:
OWNER = "owner, 2026-10-26"
REGISTER = [
    # --- APPROVED: data, period, model (owner, with the repo source of each value) ---
    dict(id="DEV_START", item="dev period, first instant (UTC)", value="2007-01-01T00:00:00+00:00", status="APPROVED",
         source=f"{OWNER}; research/sigma_compare/run_sigma.py:54 (CONF load_start); research/vol_features_r3/run_r3.py:89 (load_conf start)"),
    dict(id="DEV_END_EXCL", item="dev period, end, exclusive (UTC): data through 2020-12-31", value="2021-01-01T00:00:00+00:00",
         status="APPROVED", source=f"{OWNER}; research/vol_features_r3/run_r3.py:36 (CONF_END); research/sigma_compare/run_sigma.py:56"),
    dict(id="DATA_KEY", item="data file (manifest key)", value="database/ohlcv_data/XAUUSD.txt", status="APPROVED",
         source=f"{OWNER}; backend/datafeed/store/manifest.json:5"),
    dict(id="DATA_SHA256", item="XAUUSD.txt SHA-256 (manifest)", value="dc2d30cdee567d5043cd302ef9c4b87022d4145534fe0375dc116ae5c8dabfb9",
         status="APPROVED", source="backend/datafeed/store/manifest.json:26"),
    dict(id="LOADER", item="sealed research loader", value="research/vol_features_r3/run_r3.py load_conf", status="APPROVED",
         source=f"{OWNER} ('the repo's sealed research loader'); research/vol_features_r3/run_r3.py:89-99"),
    dict(id="CACHE_CHECK", item="the loader reads audit_long's cache, so the manifest check reaches the loaded rows only if "
         "that cache equals a fresh parse of XAUUSD.txt by the same code", value=True, status="APPROVED",
         source=f"{OWNER}: approved as the meaning of the manifest check (decision log #10); run_r3.py:90-93; "
                "research/vol_features_r2/audit_long.py:42-62"),
    dict(id="TZ", item="timestamps", value="UTC", status="APPROVED",
         source=f"{OWNER}; backend/datafeed/store/manifest.json:18; research/vol_features/DATA_AUDIT_LONG.md:11; run_r3.py:95"),
    dict(id="SIGMA_MODEL", item="HAR sigma model", value="parquet2021_2023", status="APPROVED",
         source=f"{OWNER}; backend/datafeed/features/vol_block.py:336; backend/datafeed/features/DECISIONS.md:69"),
    dict(id="SIGMA_MODEL_SHA256", item="its pinned SHA-256", value="dc0eb3c5fd31f1ee8f6e7a534b16930fbec453833aec9872b606afe963910dde",
         status="APPROVED", source="backend/datafeed/features/vol_block.py:337"),
    dict(id="HORIZONS", item="horizons H (minutes)", value=(60, 240), status="APPROVED",
         source=f"{OWNER}; backend/datafeed/features/vol_block.py:37 (only 60 and 240 fitted); research/sigma_compare/run_sigma.py:38"),
    dict(id="LICENCE", item="what committed outputs may show", value="aggregates and charts only; no raw price rows",
         status="APPROVED", source=f"{OWNER}; backend/datafeed/store/manifest.json:9"),
    # --- APPROVED: what Stage 1 describes and how calibration is scored (owner's task text) ---
    dict(id="DESCRIBE_COLUMNS", item="what step 3 describes", value=("sigma_px_60", "sigma_px_240", "sigma_ret_60", "sigma_ret_240", "L", "S_hd", "T"),
         status="APPROVED", source=f"{OWNER} (Stage 1 item 3)"),
    dict(id="DESCRIBE_GROUPS", item="how step 3 groups", value=("all", "year", "hour"), status="APPROVED",
         source=f"{OWNER} ('quantiles, by year, by hour of day')"),
    dict(id="CALIB_METHOD", item="calibration of the frozen model", value="sigma_compare post-hoc P2: mean RV/h and RMS z "
         "(and QLIKE) of HAR's h on valid, finite, non-crossing windows (posthoc.rows_from, posthoc.calib)", status="APPROVED",
         source=f"{OWNER} ('with the method from the sigma_compare study'); research/sigma_compare/POSTHOC.md:50-56; "
                "research/sigma_compare/posthoc.py:46-72"),
    dict(id="CALIB_COVERAGE", item="a window's RV counts only with >= this share of its H returns (part of CALIB_METHOD)",
         value=0.90, status="APPROVED", source="research/sigma_compare/run_sigma.py:40 (COVERAGE), used by build_context"),
    dict(id="CALIB_GROUPS", item="calibration reported", value=("all", "year"), status="APPROVED", source=f"{OWNER} ('overall and by year')"),
    # --- APPROVED 2026-10-26: Stage 1 grid and description (were PROPOSED) ---
    dict(id="EVAL_GRID", item="evaluation grid", value="every hourly label of vol_block.transform in the dev period (the close "
         "time of each 1-hour row, on the hour, UTC) that has a state by vol_block.state_at's rule (the first 120 labels of the "
         "loaded data have none); step 4 uses the subset whose H-window is valid and doesn't cross a close", status="APPROVED",
         source=f"{OWNER}: approved as proposed (decision log #7). Reasoning: HAR's state exists only at hourly labels (vol_block.py:25-31) and the model was fitted and scored on "
                "exactly this grid (run_sigma.py:149-190, posthoc.py:46-62); any finer grid would repeat the same state. "
                "Warm-up: vol_block.py:280"),
    dict(id="DESCRIBE_QUANTILES", item="quantiles reported in step 3", value=(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99),
         status="APPROVED", source=f"{OWNER}: approved (decision log #8). Reasoning: the median and quartiles for the body, 5/95 and 1/99 % for the tails that "
                                   "a squeeze/expansion study is about; no value in the repo to reuse"),
    dict(id="DESCRIBE_HOUR", item="'hour of day' in step 3", value="UTC hour of the label (the end of the hour the state describes)",
         status="APPROVED", source=f"{OWNER}: approved, with a note in the output that sessions shift by one UTC hour with DST (decision log #9). Reasoning: TZ is UTC (register); the label is the time the state is known (vol_block.py:25-31)"),
    # --- OPEN / PROPOSED: Stage 1 ---
    dict(id="L_TERCILES", item="L tercile cut-offs", value=None, status="OPEN",
         source=f"{OWNER}: stays OPEN (decision log #12). Not defined on main (docs/DECISIONS.md:265). PR #42's candidate "
                "(feat/sigma-now: backend/datafeed/features/models/l_terciles_parquet2021_2023_v1.json) was computed on the "
                "HAR model's training window, hourly labels 2021-01-03 23:00 .. 2023-12-29 21:59 UTC of the clean parquet "
                "(17,711 labels), not on this dev period. Not used by any Stage 1 cell"),
    # --- data check before steps 2-4 (owner, 2026-10-26) ---
    dict(id="DATA_CHECK", item="per year: where the daily break falls in UTC; bars on 1 Jan, 25 Dec, Good Friday; median "
         "bars per trading day", value="trade date = audit_long.session_date (18:00 New York opens the next date); season = "
         "audit_long.us_dst; break = minutes from a Mon-Thu trade date's last bar to the next trade date's first bar (its mode "
         "and share), and the modal UTC clock time of each trade date's last and first bar; holidays by UTC calendar date, "
         "Good Friday from pandas.tseries.holiday.GoodFriday; median bars over Mon-Fri trade dates", status="APPROVED",
         source=f"{OWNER} (the check's content, decision log #11); its measures as written by Claude from the repo's "
                "definitions (decision log #13, owner to confirm): research/vol_features_r2/audit_long.py:64-74"),
    dict(id="DATA_CHECK_ACCEPTED", item="the owner's decision on the data check's result (how 2007-2011, 2012-2014 and "
         "2015-2020 are treated) before steps 2-4", value=None, status="OPEN",
         source=f"{OWNER} ('If 2007-2010 differ from 2011+ ... report it and stop; don't adjust anything without my decision')"),
    # --- OPEN / PROPOSED: Q1 ---
    dict(id="Q1_TIMEFRAME", item="candle timeframe (15 or 30 min)", value=None, status="OPEN", source=f"{OWNER}"),
    dict(id="Q1_SMALL_DEF", item="a 'small' candle: range relative to sigma", value=None, status="OPEN", source=f"{OWNER}"),
    dict(id="Q1_RUN_LENGTH", item="run length that counts as a run", value=None, status="OPEN", source=f"{OWNER}"),
    dict(id="Q1_H", item="horizon H for Q1", value=None, status="OPEN", source=f"{OWNER} (HORIZONS limits it to 60 or 240)"),
    dict(id="Q1_SCORING", item="how Q1 is scored against HAR", value="sigma_compare's method: on the same valid, non-crossing "
         "rows, QLIKE of HAR's h against the same HAR with the pattern added, walk-forward with a 24 h embargo; Diebold-Mariano "
         "with Newey-West lag 8 (60 min) and BH q 0.05 across the cells; also mean RV/h and RMS z per pattern state",
         status="PROPOSED", source="reasoning: Q1 asks whether the pattern adds information beyond HAR's forecast of the move's "
                                   "size, which is what QLIKE on RV scores (research/sigma_compare/REPORT.md:12-22; "
                                   "run_sigma.py:39-46, 207-222, 402-428); NW lag and BH q as there. Lag for 240 min is part of the proposal to decide"),
    # --- OPEN: Q2 ---
    *[dict(id=i, item=t, value=None, status="OPEN", source=f"{OWNER}") for i, t in [
        ("Q2_SQZ_LOOKBACK", "squeeze lookback"), ("Q2_SQZ_THRESHOLD", "squeeze threshold"),
        ("Q2_EXPANSION", "expansion condition"), ("Q2_ENTRY_BUFFER_SIGMA", "entry buffer in sigma"),
        ("Q2_ARMING_WINDOW", "arming window"), ("Q2_STOP_RULE", "stop rule"), ("Q2_TARGET_MULT", "target multiple"),
        ("Q2_TIME_CAP", "time cap"), ("Q2_BOTH_FILLED", "both-sides-filled handling"), ("Q2_SPREAD", "spread"),
        ("Q2_STOP_SLIPPAGE", "stop-fill slippage"), ("Q2_COMMISSION", "commission"), ("Q2_NEWS", "news handling"),
        ("Q2_BENCHMARKS", "benchmarks"), ("Q2_MAX_GRID", "maximum grid size"), ("Q2_SEED", "random seed")]],
]
_REG = {r["id"]: r for r in REGISTER}
assert len(_REG) == len(REGISTER), "duplicate register id"


class RegisterError(RuntimeError):
    pass


def need(id_):
    """The value of an APPROVED register entry; raises otherwise."""
    if id_ not in _REG:
        raise RegisterError(f"{id_}: not in the register")
    r = _REG[id_]
    if r["status"] != "APPROVED":
        raise RegisterError(f"{id_} is {r['status']}: {r['item']} (value {r['value']!r}; {r['source']})")
    return r["value"]


def blocked(*ids):
    """The ids among these that aren't APPROVED."""
    return [i for i in ids if _REG[i]["status"] != "APPROVED"]


reg = pd.DataFrame(REGISTER)[["id", "status", "item", "value", "source"]]
print(reg["status"].value_counts().to_string())
reg

status
OPEN        22
APPROVED    20
PROPOSED     1


,id,status,item,value,source
0,DEV_START,APPROVED,"dev period, first instant (UTC)",2007-01-01T00:00:00+00:00,"owner, 2026-10-26; research/sigma_compare/run_sigma.py:54 (CONF load_start); research/vol_features_r3/run_r3.py:89 (load_conf start)"
1,DEV_END_EXCL,APPROVED,"dev period, end, exclusive (UTC): data through 2020-12-31",2021-01-01T00:00:00+00:00,"owner, 2026-10-26; research/vol_features_r3/run_r3.py:36 (CONF_END); research/sigma_compare/run_sigma.py:56"
2,DATA_KEY,APPROVED,data file (manifest key),database/ohlcv_data/XAUUSD.txt,"owner, 2026-10-26; backend/datafeed/store/manifest.json:5"
3,DATA_SHA256,APPROVED,XAUUSD.txt SHA-256 (manifest),dc2d30cdee567d5043cd302ef9c4b87022d4145534fe0375dc116ae5c8dabfb9,backend/datafeed/store/manifest.json:26
4,LOADER,APPROVED,sealed research loader,research/vol_features_r3/run_r3.py load_conf,"owner, 2026-10-26 ('the repo's sealed research loader'); research/vol_features_r3/run_r3.py:89-99"
5,CACHE_CHECK,APPROVED,"the loader reads audit_long's cache, so the manifest check reaches the loaded rows only if that cache equals a fresh parse of XAUUSD.txt by the same code",True,"owner, 2026-10-26: approved as the meaning of the manifest check (decision log #10); run_r3.py:90-93; research/vol_features_r2/audit_long.py:42-62"
6,TZ,APPROVED,timestamps,UTC,"owner, 2026-10-26; backend/datafeed/store/manifest.json:18; research/vol_features/DATA_AUDIT_LONG.md:11; run_r3.py:95"
7,SIGMA_MODEL,APPROVED,HAR sigma model,parquet2021_2023,"owner, 2026-10-26; backend/datafeed/features/vol_block.py:336; backend/datafeed/features/DECISIONS.md:69"
8,SIGMA_MODEL_SHA256,APPROVED,its pinned SHA-256,dc0eb3c5fd31f1ee8f6e7a534b16930fbec453833aec9872b606afe963910dde,backend/datafeed/features/vol_block.py:337
9,HORIZONS,APPROVED,horizons H (minutes),"(60, 240)","owner, 2026-10-26; backend/datafeed/features/vol_block.py:37 (only 60 and 240 fitted); research/sigma_compare/run_sigma.py:38"


## Context: what the repo's reports say (every claim CITED: file:line, checked 2026-10-26)

**The vol block (L, S_hd, T).**
- CITED: L, S and T beat the 13 earlier volatility features at forecasting deseasonalized future variance, on XAUUSD
  2007–2020: out-of-sample R² 0.73 vs 0.52 (4 h) and 0.80 vs 0.55 (24 h), 24/24 half-years
  (`research/vol_block/VOL_BLOCK_DECISION.md:11-15`). The signs L +, S +, T − hold in every fold
  (`VOL_BLOCK_DECISION.md:16-17`).
- CITED: **S_hd** (S minus its training-window hour mean) is an engineering decision; its out-of-sample effect was only
  measured exploratorily (`VOL_BLOCK_DECISION.md:31-34`). R2 (no clock) failed narrowly in round 3
  (`VOL_BLOCK_DECISION.md:23-27`).

**HAR vs the plug-in (sigma_compare).**
- CITED: on CONF 2007–2020, non-crossing windows: QLIKE 0.2493 vs 0.3044 at 60 min (n 68,040) and 0.1838 vs 0.2190 at
  240 min (n 58,920) (`research/sigma_compare/REPORT.md:17-18`): −18 % and −16 % (`REPORT.md:20`), better in
  24/24 folds (`REPORT.md:22`).
- CITED + INFERRED: **this notebook's dev period is that study's CONF period.** The HAR *form* was chosen on it, although the pinned
  model's coefficients and VolParams were fitted on 2021–2023 only
  (`backend/datafeed/features/models/sigma_har_parquet2021_2023_v1.json:657-658`: train 2021-01-03 .. 2023-12-29).
  So the dev period is seen data for the method, unseen for this model's fit.

**Out of era.**
- CITED: frozen on `XAUUSD.txt` 2007–2020 and scored without refit on the parquet 2021–2023: QLIKE 0.2142 vs 0.2722
  (60 min, n 16,921) and 0.1601 vs 0.1987 (240 min, n 14,608), −21 % and −19 %
  (`research/sigma_compare/POSTHOC_RESULT.md:18-21`).

**HAR calibration.**
- CITED: walk-forward on DEV 2022–2023, the parquet recipe (the pinned model's, "SHORT"): mean RV/h 0.994 (60) and 0.982
  (240), RMS z 0.968 and 0.969 (`POSTHOC_RESULT.md:36,38`), inside the [0.9, 1.1] band (`POSTHOC.md:60`).
- CITED: **a difference to note:** the post-hoc rule favoured the txt fit (LONG) as the default
  (`POSTHOC_RESULT.md:8-9,41`). The owner later made `parquet2021_2023` the default, on QLIKE
  (`backend/datafeed/features/DECISIONS.md:69-76`). That is the model this notebook uses (register SIGMA_MODEL).
- INFERRED (from a search of research/ and docs/): the calibration of this model on 2007–2020 (step 4) has not been reported anywhere yet.

**Relevant nulls.**
- CITED: **levels (2007–2020 CONF):** 0 of 6 hypotheses supported; daily-level breaks continued *less* than placebo
  breaks, −7.3 pp [−9.6, −5.0] (`research/levels/REPORT.md:19`).
- CITED: **levels_confirm (2024-01-01 .. 2026-01-30):** REJECTED; near-open PDH/PDL breaks continued *more* than
  distance-matched placebos, +11.3 pp [+4.0, +18.5] (`research/levels_confirm/REPORT.md:10,18`). INFERRED: the two
  break studies disagree in sign across eras, and neither supports a usable rule.
- **Asia → London continuation: not established (legacy notebook cell only).** CITED: there is no study or report in the repo. The only analysis is an exploratory
  cell of the legacy notebook, `legacy/notebooks/backtest.ipynb` (cell 16, 2022-01-01 .. 2026-01-30 per cell 3):
  its saved output reads "Overall continuation rate: 52.3 % (685 / 1311 days)", with filtered subsets at
  55.7–58.8 % and no test, no PREREG and no placebo. INFERRED: it is not a null result in the repo's sense.
- No numbers were quoted in the request, so there were none to check against the reports.

**What HAR does and doesn't say.** CITED: HAR forecasts the *variance* of the log return over the next H market-open
minutes, `var(t, H) = P(t, H) × exp(β · [1, L, S_hd, T]) × c` (`backend/datafeed/features/vol_block.py:33`).
INFERRED from that definition: **it predicts the size of moves, not their direction, and not whether a move persists.**
INFERRED (from a search of research/): no study in the repo tested direction or persistence from it.

## Decision log (append-only)

| # | date | id | decision | by |
|---|---|---|---|---|
| 1 | 2026-10-26 | (all) | Notebook, register rule, Q1/Q2 framing, the APPROVED items DEV_START, DEV_END_EXCL, DATA_KEY, LOADER, TZ, SIGMA_MODEL, HORIZONS, LICENCE, DESCRIBE_COLUMNS, DESCRIBE_GROUPS, CALIB_METHOD, CALIB_GROUPS, and the OPEN list | owner |
| 2 | 2026-10-26 | (branch) | `research/squeeze-oco` is stacked on `feat/data-manifest` (PR #41): the manifest check exists only there | Claude |
| 3 | 2026-10-26 | CACHE_CHECK | "Through the manifest check" read as: verify XAUUSD.txt's SHA-256, and check that the loader's cache equals a fresh parse of it by the loader's own code, since `load_conf` reads the cache, not the file. Owner to confirm | Claude |
| 4 | 2026-10-26 | CALIB_COVERAGE, DATA_SHA256, SIGMA_MODEL_SHA256 | Taken as part of the approved items they belong to (sigma_compare's method; the manifest; the pinned model), with their file:line | Claude |
| 5 | 2026-10-26 | EVAL_GRID | Found while testing: over a whole frame, `vol_block.transform` gives finite L, S_hd, T for the first 120 hourly labels, whose 120-hour window reaches the first (possibly partial) row; `state_at` refuses them. The grid proposal follows `state_at` (`stage1.features`) | Claude |
| 6 | 2026-10-26 | (context) | Asia → London continuation: no report in the repo; only the legacy notebook cell, cited as such | Claude |
| 7 | 2026-10-26 | EVAL_GRID | APPROVED as proposed | owner |
| 8 | 2026-10-26 | DESCRIBE_QUANTILES | APPROVED: 1, 5, 25, 50, 75, 95, 99 % | owner |
| 9 | 2026-10-26 | DESCRIBE_HOUR | APPROVED: the UTC hour; the output notes that sessions shift by one UTC hour with DST | owner |
| 10 | 2026-10-26 | CACHE_CHECK | "The loader's cache equals a fresh parse" APPROVED as the meaning of the manifest check | owner |
| 11 | 2026-10-26 | DATA_CHECK | Before steps 2–4: per year 2007–2020, where the daily break falls in UTC, bars on CME holidays (1 Jan, 25 Dec, Good Friday), median bars per trading day. If 2007–2010 differ from 2011+ (feed, timezone or calendar): report and stop, adjust nothing | owner |
| 12 | 2026-10-26 | L_TERCILES, (context) | L_TERCILES stays OPEN. Asia → London cited as "not established (legacy notebook cell only)" | owner |
| 13 | 2026-10-26 | DATA_CHECK | The check's measures written from the repo's definitions (audit_long.session_date, us_dst; pandas' Good Friday); the break measured as the gap to the next trade date's first bar, since the first bar at 18:00 New York follows from the trade-date cut itself. Owner to confirm | Claude |
| 14 | 2026-10-26 | DATA_CHECK_ACCEPTED | The check shows three regimes (see "Data check: result"). By decision #11, steps 2–4 are stopped; DATA_CHECK_ACCEPTED added as OPEN | Claude |
| 15 | 2026-10-26 | (all) | Overnight rules: never fill an OPEN item or promote a PROPOSED one; label every claim OBSERVED, CITED or INFERRED (UNVERIFIED if it can't be checked); stop Stage 1 if the data check fails and continue with the safe tasks | owner |


In [3]:
a = stage1.attempts()
print(f"outcome analyses so far: {a['outcome_analyses']}  (Stage 1 is descriptive and doesn't count)")

outcome analyses so far: 0  (Stage 1 is descriptive and doesn't count)


## Guard: the dev period only

`stage1.dev_guard` refuses any timestamp outside [DEV_START, DEV_END_EXCL). The upper side is the repo's
guards: `research/sealing.py` `assert_before_global_seal` (nothing from 2026-10-01) and `run_r3.assert_sealed`
with the study end. The repo has no lower-side guard, so that side is one comparison. Every frame below goes
through it; this cell shows that it refuses.

In [4]:
DEV_START, DEV_END = pd.Timestamp(need("DEV_START")), pd.Timestamp(need("DEV_END_EXCL"))
for bad in ["2006-12-31 23:59", "2021-01-01 00:00", "2026-10-01 00:00"]:
    try:
        stage1.dev_guard(pd.DatetimeIndex([bad], tz="UTC"), DEV_START, DEV_END, "probe")
        raise SystemExit(f"guard let {bad} through")
    except AssertionError as e:
        print(f"refused {bad}: {str(e)[:90]}")
stage1.dev_guard(pd.DatetimeIndex(["2007-01-01 00:00", "2020-12-31 23:59"], tz="UTC"), DEV_START, DEV_END, "probe")
print("accepted 2007-01-01 00:00 and 2020-12-31 23:59")

refused 2006-12-31 23:59: OUTSIDE THE DEV PERIOD: probe has rows before 2007-01-01 00:00:00+00:00
refused 2021-01-01 00:00: SEALED DATA: probe has rows at or after 2021-01-01
refused 2026-10-01 00:00: SEALED DATA: probe contains timestamps >= 2026-10-01 (reserved for the final strategy test
accepted 2007-01-01 00:00 and 2020-12-31 23:59


## Stage 1, step 1: load the dev-period bars

The data guard (`research/data_guard.py`), the manifest's SHA-256 of `XAUUSD.txt`, the cache check, then
`run_r3.load_conf` for [DEV_START, DEV_END_EXCL), through the guard. The model is loaded with its SHA-256 checked.
Shown: aggregates per year only.

In [5]:
assert need("TZ") == "UTC" and need("LOADER") == "research/vol_features_r3/run_r3.py load_conf"
m, checks = stage1.load_dev(need("DATA_KEY"), DEV_START, DEV_END, need("CACHE_CHECK"))
assert checks["manifest_sha256"] == need("DATA_SHA256")
model = stage1.load_model(need("SIGMA_MODEL"), need("SIGMA_MODEL_SHA256"))
print(checks)
print(f"model {model.name}: SHA-256 {model.sha256} checked; horizons {sorted(model.horizons)}")
assert tuple(sorted(model.horizons)) == tuple(need("HORIZONS"))
print(f"bars: {len(m):,}, {m.index.min()} .. {m.index.max()} (UTC)")
cov = stage1.coverage_table(m)
cov

{'manifest_key': 'database/ohlcv_data/XAUUSD.txt', 'manifest_sha256': 'dc2d30cdee567d5043cd302ef9c4b87022d4145534fe0375dc116ae5c8dabfb9', 'cache_equals_fresh_parse': True, 'cache_rows': 6971972}
model parquet2021_2023: SHA-256 dc0eb3c5fd31f1ee8f6e7a534b16930fbec453833aec9872b606afe963910dde checked; horizons [60, 240]
bars: 4,983,505, 2007-01-01 00:01:00+00:00 .. 2020-12-31 21:59:00+00:00 (UTC)


,bars,days_with_bars,first_bar_utc,last_bar_utc
year,,,,
2007,346941,313,2007-01-01 00:01:00+00:00,2007-12-31 23:57:00+00:00
2008,363465,314,2008-01-01 00:00:00+00:00,2008-12-31 23:56:00+00:00
2009,363585,313,2009-01-01 00:03:00+00:00,2009-12-31 23:56:00+00:00
2010,365764,313,2010-01-01 00:02:00+00:00,2010-12-31 21:59:00+00:00
2011,363600,311,2011-01-02 22:00:00+00:00,2011-12-30 21:59:00+00:00
2012,353161,312,2012-01-02 23:00:00+00:00,2012-12-31 21:59:00+00:00
2013,352468,311,2013-01-01 23:00:00+00:00,2013-12-31 21:59:00+00:00
2014,351953,311,2014-01-01 23:00:00+00:00,2014-12-31 21:59:00+00:00
2015,353860,311,2015-01-01 23:00:00+00:00,2015-12-31 21:59:00+00:00


## Stage 1, data check (before steps 2–4)

Per year 2007–2020 (register DATA_CHECK): where the daily break falls in UTC, bars on 1 January, 25 December and
Good Friday, and the median bars per trading day. If 2007–2010 differ from 2011+, Stage 1 stops here until the
owner decides (DATA_CHECK_ACCEPTED). Nothing is adjusted.

In [6]:
assert need("DATA_CHECK")
brk = stage1.break_table(m)
brk.round(3)

trade_dates  break_min_mode  break_share last_bar_mode_utc  last_share first_bar_mode_utc  first_share
year season                                                                                                        
2007 DST             204             1.0        0.287             21:59       0.319              22:00        0.662
     STD             109             1.0        0.425             22:59       0.376              23:00        0.771
2008 DST             204             1.0        0.493             21:59       0.407              22:00        0.814
     STD             110             1.0        0.473             22:59       0.391              23:00        0.836
2009 DST             204             1.0        0.507             21:59       0.407              22:00        0.833
     STD             109             1.0        0.534             22:59       0.422              23:00        0.826
2010 DST             204             1.0        0.868             21:59       0.652              22:00        0.833
     STD             109             1.0        0.736             22:59       0.596              23:00        0.835
2011 DST             204             1.0        0.485             21:59       0.397              22:00        0.833
     STD             104             1.0        0.423             22:59       0.346              23:00        0.846
2012 DST             170            60.0        0.728             21:00       0.588              22:00        0.965
     STD              89            60.0        0.732             22:00       0.584              23:00        0.955
2013 DST             169            60.0        0.588             21:00       0.521              22:00        0.899
     STD              89            60.0        0.831             22:00       0.674              23:00        0.955
2014 DST             170            60.0        0.632             21:00       0.506              22:00        0.859
     STD              89            60.0        0.831             22:00       0.663              23:00        0.966
2015 DST             169            61.0        0.537             20:59       0.609              22:00        0.970
     STD              89            61.0        0.611             21:59       0.685              23:00        0.989
2016 DST             169            61.0        0.890             20:59       0.911              22:00        1.000
     STD              89            61.0        0.775             21:59       0.809              23:00        1.000
2017 DST             169            61.0        0.897             20:59       0.852              22:00        0.994
     STD              88            61.0        0.486             21:59       0.500              23:00        0.977
2018 DST             169            61.0        0.699             20:59       0.586              22:00        1.000
     STD              89            61.0        0.915             21:59       0.899              23:00        1.000
2019 DST             169            61.0        0.537             20:59       0.503              22:00        0.988
     STD              89            61.0        0.915             21:59       0.933              23:00        1.000
2020 DST             169            61.0        0.978             20:59       0.982              22:00        0.994
     STD              90            61.0        0.944             21:59       0.944              23:00        1.000

In [7]:
hol = stage1.holiday_table(m, range(DEV_START.year, DEV_END.year))
hol

,bars_jan_1,bars_dec_25,good_friday,bars_good_friday,holiday_bars_at_23_utc,holiday_bars_other_hours,bars_on_utc_saturdays
year,,,,,,,
2007,604,696,2007-04-06,512,99,1713,0
2008,690,655,2008-03-21,578,82,1841,0
2009,715,635,2009-04-10,662,59,1953,0
2010,583,0,2010-04-02,996,0,1579,0
2011,0,0,2011-04-22,818,0,818,0
2012,0,53,2012-04-06,0,53,0,0
2013,59,0,2013-03-29,0,59,0,0
2014,60,59,2014-04-18,0,119,0,0
2015,60,0,2015-04-03,0,60,0,0


In [8]:
bpd = stage1.bars_per_trade_date(m)
bpd

,trade_dates_mon_fri,median_bars_per_trade_date,weekend_trade_dates
timestamp,,,
2007,261,1335,52
2008,262,1396,52
2009,261,1403,52
2010,261,1412,52
2011,259,1411,49
2012,259,1379,0
2013,258,1378,0
2014,258,1380,1
2015,258,1380,0


## Data check: result. **2007–2011 differ from 2012+; Stage 1 stops here** (decision log #11, #14)

Every number below is from the three tables above.

**Daily break (`brk`).**
- OBSERVED: in **2007–2011** the most common gap from a trade date's last bar to the next one's first bar is
  **1 minute**, in both seasons (28.7–86.8 % of those gaps): **no daily break**. The last bar sits at 21:59 /
  22:59 UTC (DST / STD) and the next session's first bar one minute later.
- OBSERVED: in **2012–2014** the gap is **60 minutes**: the last bar at 21:00 / 22:00 UTC, the next first bar at
  22:00 / 23:00 UTC.
- OBSERVED: in **2015–2020** it is **61 minutes**: the last bar at 20:59 / 21:59 UTC, the next first bar at
  22:00 / 23:00 UTC.
- INFERRED: in every year the edges move by one UTC hour between DST and STD, so the session is anchored to
  New York time and the timestamps are UTC throughout. This is not a timezone difference: the break exists from
  2012 and not before, and its last bar moves by one minute between 2014 and 2015.

**Holidays (`hol`).**
- OBSERVED: **2007–2010** have 512–996 bars on each of 1 January, 25 December and Good Friday (except 25 December
  2010: 0), mostly outside 23:00 UTC (1,579–1,953 a year). **2011** has 818 bars on Good Friday and none on the
  other two dates.
- OBSERVED: from **2012**, every holiday bar is in the 23:00 UTC hour (0–120 a year), and none at other hours.
  Good Friday has 0 bars in every year 2012–2020.
- INFERRED: 23:00 UTC is where the winter session for the next date opens, so 2012+ has no trading on these
  holidays; 2007–2011 trades through them.
- OBSERVED: no year has a bar on a UTC Saturday.

**Bars per trading day (`bpd`).**
- OBSERVED: the median bars per Mon–Fri trade date is **1,335 / 1,396 / 1,403 / 1,412 / 1,411** in 2007–2011, and
  **1,377–1,380** in 2012–2020.
- OBSERVED: 2007–2010 have **52** trade dates a year that fall on a Saturday, and 2011 has **49**. 2012–2020 have
  0, except 2014 with 1.
- INFERRED: there are no UTC-Saturday bars, so those trade dates are bars after Friday 17:00 New York (the trade
  date rolls at 18:00). The old feed trades Friday evening; from 2012 the week closes at 17:00 New York.
- INFERRED: 1,380 = 23 h × 60 min, the CME Globex day. Medians above it in 2008–2011 fit a day with no break.

**What it means (INFERRED).** 2007–2011 look like a different feed or session convention (24-hour trading with no
daily break, trading on CME holidays and on Friday evenings). 2012–2014 and 2015–2020 also differ from each other,
by the one-minute shift of the last bar before the break. Stage 1 steps 2–4 are stopped until the owner decides
(DATA_CHECK_ACCEPTED, OPEN). Nothing was adjusted, excluded or re-read.


## Stage 1, steps 2–3: the vol state on the evaluation grid, described

Runs only once DATA_CHECK_ACCEPTED, EVAL_GRID, DESCRIBE_QUANTILES and DESCRIBE_HOUR are approved. The hour of day is
the UTC hour of the label; sessions shift by one UTC hour with US DST.

In [9]:
b = blocked("DATA_CHECK_ACCEPTED", "EVAL_GRID", "DESCRIBE_QUANTILES", "DESCRIBE_HOUR")
if b:
    print("BLOCKED by:", ", ".join(f"{i} ({_REG[i]['status']})" for i in b))
    tbl = None
else:
    ctx = stage1.context(m, DEV_END, need("HORIZONS"), need("CALIB_COVERAGE"))
    tbl = stage1.state_table(ctx, model, DEV_START, DEV_END)
    cols, q = list(need("DESCRIBE_COLUMNS")), list(need("DESCRIBE_QUANTILES"))
    assert need("DESCRIBE_HOUR").startswith("UTC hour")
    print("Note: the hour of day is the UTC hour. Sessions (and the daily break) shift by one UTC hour with US DST.")
    print(f"grid: {len(tbl):,} hourly labels with a state")
    for g in need("DESCRIBE_GROUPS"):
        print(f"--- {g}")
        display(stage1.describe(tbl, cols, q, None if g == "all" else g))

BLOCKED by: DATA_CHECK_ACCEPTED (OPEN)


## Stage 1, step 4: calibration on the dev period (the pinned model never saw it)

sigma_compare's post-hoc P2 metrics (mean RV/h, RMS z, QLIKE) of the frozen model on valid, non-crossing
windows, overall and by year. Runs only once DATA_CHECK_ACCEPTED and EVAL_GRID are approved.

In [10]:
b = blocked("DATA_CHECK_ACCEPTED", "EVAL_GRID")
if b:
    print("BLOCKED by:", ", ".join(f"{i} ({_REG[i]['status']})" for i in b))
else:
    assert need("CALIB_GROUPS") == ("all", "year")
    cal = stage1.calibration(ctx, model, DEV_START, DEV_END, True)
    display(cal)

BLOCKED by: DATA_CHECK_ACCEPTED (OPEN)


## What Stage 1 found (run 2026-10-26)

- OBSERVED (step 1): `XAUUSD.txt` matches its manifest SHA-256 (`dc2d30cd…`). The loader's cache equals a fresh
  parse of it (6,971,972 rows before 2024).
- OBSERVED (step 1): the dev period loaded through `run_r3.load_conf`: **4,983,505** 1-min bars, 2007-01-01 00:01
  .. 2020-12-31 21:59 UTC; per year 346,941–365,764 bars on 309–314 days with bars.
- OBSERVED (guard): it refuses 2006-12-31 23:59, 2021-01-01 00:00 and 2026-10-01 00:00, and accepts the period's
  edges.
- OBSERVED (step 1): `parquet2021_2023` loads with its pinned SHA-256 (`dc0eb3c5…`), horizons 60 and 240.
- OBSERVED (data check): **2007–2011 differ from 2012+.** They have no daily break, trade on 1 January, 25 December
  and Good Friday, and have Friday-evening sessions. 2012–2014 differ from 2015–2020 by one minute at the break.
  Details in "Data check: result".
- **Not run:** steps 2–4 (vol state, its distributions, calibration). They are blocked by DATA_CHECK_ACCEPTED
  (OPEN), as the owner's rule for this check says. No sigma, L, S_hd, T or calibration number has been computed.
- OBSERVED: outcome analyses 0 (`ATTEMPTS.json`).


## Plan (not filled in)

- **Q1 options:** `PROPOSALS.md` (options for the OPEN Q1 items, with event counts; not run).
- **Stage 2, Q1:** the small-candle pattern against HAR, once its parameters are approved (Q1_TIMEFRAME,
  Q1_SMALL_DEF, Q1_RUN_LENGTH, Q1_H, Q1_SCORING). It is the first outcome analysis, so it increments `ATTEMPTS.json`.
- **Stage 3:** the squeeze definition and event counts.
- **Stage 4:** an order and fill simulator, tested on synthetic data.
- **Stage 5:** PREREG.
- **Stage 6:** one look at untouched data.